# CODY-SAM3 — SAM 3 extraction on the external inference datasets (Colab)

Adapted from the training extraction notebook (phase 1a). **The processing functions (segmentation, contour/grid sampling, geometric descriptors) are taken VERBATIM** from the training notebook — this is essential so that the inference features remain strictly consistent with what the Tier 2 models saw at training time.

**What changes vs the training notebook**: Colab installation, download of the `sam3.pt` weights from Hugging Face (access must be approved), Google Drive paths, `.wmv` support, a loop over the 3 external datasets, and outputs written directly to Drive.

**Requirements**:
- GPU runtime (ideally A100/L4): `Runtime > Change runtime type > GPU`
- Your Hugging Face token (Settings -> Access Tokens on huggingface.co)
- Videos on Drive under `dataset_inference/dataset_X/P<n>/video.ext`

**Expected video layout**:
```
MyDrive/sam_3_infer/dataset_inference/
  dataset_1/  P28/<video>  P29/<video>
  dataset_2/  P1/<video> ... P12/<video>
  dataset_3/  P1/<video> ... P20/<video>
```

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Installation

Ultralytics >= 8.3.237 (SAM 3), and the **correct** `clip` package (otherwise prediction fails with `SimpleTokenizer object is not callable`).

In [ ]:
# Ultralytics avec support SAM 3 (>= 8.3.237)
!pip install -q -U "ultralytics>=8.3.237"

# Fix du package clip (Colab a souvent le mauvais 'clip' preinstalle)
!pip uninstall -y clip >/dev/null 2>&1
!pip install -q git+https://github.com/ultralytics/CLIP.git

# huggingface_hub to download the weights
!pip install -q -U huggingface_hub

# opencv headless + scipy (normally already present on Colab)
!pip install -q opencv-python-headless scipy
print('Install OK')

## 3. Imports and runtime check

In [ ]:
import os
import math
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from scipy.spatial import distance
from scipy.ndimage import center_of_mass

import torch

import logging
logging.getLogger('ultralytics').setLevel(logging.ERROR)

from ultralytics.models.sam import SAM3SemanticPredictor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM:           {vram_gb:.1f} GB')
print(f'Device used:    {DEVICE}')

## 4. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Download the SAM 3 weights (~3.45 GB)

Requires approved access to `facebook/sam3` (request it on the Hugging Face model page). Paste your **Hugging Face token** when prompted (it will not be echoed). The download takes a few minutes.

In [ ]:
from huggingface_hub import login, hf_hub_download
from getpass import getpass

# Login (the token is not echoed). Alternative: set the HF_TOKEN environment variable.
hf_token = os.environ.get('HF_TOKEN') or getpass('HuggingFace token: ')
login(token=hf_token)

# Download sam3.pt into /content (fast, ephemeral)
MODEL_PATH = Path('/content/sam3.pt')
if not MODEL_PATH.exists():
    print('Downloading sam3.pt from facebook/sam3 ...')
    p = hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt',
                        local_dir='/content', local_dir_use_symlinks=False)
    print('Downloaded ->', p)
else:
    print('sam3.pt already present:', MODEL_PATH)
print(f'Taille: {MODEL_PATH.stat().st_size/1024**3:.2f} GB')

## 6. Configuration

**Important**: the sampling parameters (`n_contour_points=64`, `grid_rows=12`, `grid_cols=8`, `imgsz=770`, `text_prompt='person'`) are **identical to training**. Do not change them, otherwise the features will be incompatible with the Tier 2 models.

In [ ]:
# ----- Drive paths (adapt if needed) -----
DRIVE_ROOT  = Path('/content/drive/MyDrive/sam_3_infer')
INFER_ROOT  = DRIVE_ROOT / 'dataset_inference'   # contient dataset_1/2/3
OUTPUT_ROOT = DRIVE_ROOT / 'outputs_inference'    # sorties SAM 3 (persistant)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

@dataclass
class Settings:
    text_prompt:        str   = 'person'
    confidence:         float = 0.25
    imgsz:              int   = 770        # multiple de 14 (stride backbone SAM 3)
    half:               bool  = True
    n_contour_points:   int   = 64
    grid_rows:          int   = 12
    grid_cols:          int   = 8
    auto_select_largest:    bool  = True
    iou_continuity_thresh:  float = 0.10
    manual_bbox:        Optional[Tuple[int, int, int, int]] = None
    frame_stride:       int   = 1

S = Settings()
print(S)

## 7. Dataset discovery (recursive, per patient)

`.wmv` added to the accepted extensions. Warning: if a `.wmv` video cannot be decoded on Colab (codec), a clear error message is printed — convert it to `.mp4` with ffmpeg beforehand.

In [ ]:
import re

# Accepted video extensions (case-insensitive). Add more here if needed.
VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v', '.wmv'}

@dataclass
class VideoEntry:
    path:          Path
    relative_path: str            # e.g. 'P3/posture.mp4'
    subject_id:    str            # 'P3', 'C1', ...
    subject_type:  str            # 'patient' / 'control' / 'other'
    subject_num:   Optional[int]  # 3, 1, ...
    manual_bbox:   Optional[Tuple[int, int, int, int]] = None

# Accept folder names like 'P1', 'P1_RPA', 'P12-v2', 'C3', 'C3_session1', etc.
# The mandatory prefix is 'P<n>' (patient) or 'C<n>' (control); anything that
# follows the integer (after an underscore or a dash) is treated as an optional
# descriptive suffix and ignored for identification.
_SUBJECT_RE = re.compile(r'^([PC])(\d+)(?:[_\-].*)?$', re.IGNORECASE)

def parse_subject_folder(name: str) -> Tuple[str, Optional[int]]:
    """Return (subject_type, subject_num) from a folder name.

    Accepted patterns (case-insensitive):
        P1, P12, P1_RPA, P12-v2, P3_session1
        C1, C12, C1_RPA, C12-v2, C3_session1
    The integer right after 'P' or 'C' is the subject number; any '_<suffix>'
    or '-<suffix>' that follows is recorded but does not affect identification.
    """
    m = _SUBJECT_RE.match(name)
    if not m:
        return 'other', None
    prefix, num = m.group(1).upper(), int(m.group(2))
    return ('patient' if prefix == 'P' else 'control'), num

def list_videos_in(folder: Path) -> List[Path]:
    """Case-insensitive multi-extension listing, alphabetically sorted."""
    out = [
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in VIDEO_EXTS
        and not p.name.startswith('.')   # skip hidden / macOS sidecar files
    ]
    return sorted(out, key=lambda p: p.name.lower())

def load_manual_targets(csv_path: Path) -> dict:
    """Return {relative_path: (xmin, ymin, xmax, ymax)} from manual_targets.csv."""
    if not csv_path.exists():
        return {}
    df_t = pd.read_csv(csv_path)
    needed = {'relative_path', 'bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax'}
    if not needed.issubset(df_t.columns):
        print(f'WARNING: {csv_path} is missing required columns; ignoring it.')
        return {}
    out = {}
    for _, r in df_t.iterrows():
        out[str(r['relative_path']).replace('\\', '/')] = (
            int(r['bbox_xmin']), int(r['bbox_ymin']),
            int(r['bbox_xmax']), int(r['bbox_ymax']),
        )
    return out

def discover_dataset(
    dataset_dir:    Path,
    manual_targets: Optional[dict] = None,
) -> List[VideoEntry]:
    """Walk dataset_dir, parse subject folders, return one VideoEntry per video."""
    if manual_targets is None:
        manual_targets = {}

    entries: List[VideoEntry] = []
    for subject_dir in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
        subject_id        = subject_dir.name
        subject_type, num = parse_subject_folder(subject_id)
        if subject_type == 'other':
            print(f'WARNING: folder "{subject_id}" does not match P<n>[_suffix] / C<n>[_suffix] -> skipped')
            continue

        # Warn (not crash) if the user has nested subfolders inside a subject folder.
        nested = [p for p in subject_dir.iterdir() if p.is_dir()]
        if nested:
            print(f'WARNING: {subject_dir.name} has subfolder(s) {[n.name for n in nested]} '
                  f'that are NOT scanned (videos must be directly inside {subject_dir.name}/).')

        videos = list_videos_in(subject_dir)
        if not videos:
            print(f'WARNING: no video file in {subject_dir} -> skipped')
            continue
        for v in videos:
            rel = f'{subject_id}/{v.name}'
            entries.append(VideoEntry(
                path          = v,
                relative_path = rel,
                subject_id    = subject_id,
                subject_type  = subject_type,
                subject_num   = num,
                manual_bbox   = manual_targets.get(rel),
            ))
    return entries

def print_dataset_summary(entries: List[VideoEntry]) -> None:
    """Per-subject summary table: count and list of filenames."""
    if not entries:
        print('No videos discovered.'); return

    rows = []
    by_subject = {}
    for e in entries:
        by_subject.setdefault(e.subject_id, []).append(e)

    # Sort subjects: patients first, controls second; within each group, by numeric id.
    _type_priority = {'patient': 0, 'control': 1, 'other': 2}
    def _key(sid):
        es = by_subject[sid]
        return (
            _type_priority.get(es[0].subject_type, 9),
            es[0].subject_num if es[0].subject_num is not None else 10**9,
        )
    subj_order = sorted(by_subject.keys(), key=_key)

    width_id    = max(8,  max(len(s) for s in subj_order))
    width_files = max(40, max(sum(len(e.path.name) + 2 for e in by_subject[s]) for s in subj_order))
    header = f'{"subject":<{width_id}}  {"type":<8}  {"#vids":>5}  filenames'
    print(header)
    print('-' * (len(header) + 20))
    for sid in subj_order:
        es     = by_subject[sid]
        names  = ', '.join(e.path.name for e in es)
        n_man  = sum(e.manual_bbox is not None for e in es)
        suffix = f'  [MANUAL x {n_man}]' if n_man else ''
        print(f'{sid:<{width_id}}  {es[0].subject_type:<8}  {len(es):>5}  {names}{suffix}')

    n_pat  = sum(e.subject_type == 'patient' for e in entries)
    n_ctrl = sum(e.subject_type == 'control' for e in entries)
    n_man  = sum(e.manual_bbox is not None for e in entries)
    print('-' * (len(header) + 20))
    print(f'TOTAL: {len(entries)} videos across {len(by_subject)} subjects '
          f'({n_pat} patient + {n_ctrl} control rows; {n_man} with manual override).')


## 8. Charger SAM 3

In [ ]:
overrides = dict(
    conf       = S.confidence,
    task       = 'segment',
    mode       = 'predict',
    model      = str(MODEL_PATH),
    half       = S.half,
    imgsz      = S.imgsz,
    save       = False,
    verbose    = False,
    device     = 0 if DEVICE == 'cuda' else 'cpu',
)
predictor = SAM3SemanticPredictor(overrides=overrides)
print('SAM 3 predictor ready.')

## 9. Processing functions (VERBATIM from the training notebook)

These cells are copied **unchanged**. Do not modify.

In [ ]:
def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-Union between two boolean masks of the same shape."""
    if a is None or b is None:
        return 0.0
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union > 0 else 0.0

def mask_centroid(mask: np.ndarray) -> Tuple[float, float]:
    """Return the (x, y) centroid of a boolean mask. NaN if mask is empty."""
    if mask.sum() == 0:
        return (float('nan'), float('nan'))
    cy, cx = center_of_mass(mask.astype(np.uint8))
    return float(cx), float(cy)

def select_patient_mask(
    masks:        np.ndarray,             # shape (K, H, W) bool
    boxes:        np.ndarray,             # shape (K, 4) xyxy
    prev_mask:    Optional[np.ndarray],   # last frame's patient mask, or None
    cfg:          Settings,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Choose ONE mask among the K candidates returned by SAM 3.

    Returns
    -------
    mask_sel : (H, W) bool, or None if no plausible patient was found
    box_sel  : (4,) np.ndarray xyxy, or None
    """
    if masks is None or len(masks) == 0:
        return None, None

    # Cast to bool
    masks = masks.astype(bool)

    # ------------------------------------------------------------------
    # FRAME 0 case (no previous mask)
    # ------------------------------------------------------------------
    if prev_mask is None:
        # Manual bbox override?
        if cfg.manual_bbox is not None:
            xmin, ymin, xmax, ymax = cfg.manual_bbox
            best_idx, best_iou = -1, -1
            for i, b in enumerate(boxes):
                # IoU between cfg.manual_bbox and candidate box
                xa = max(b[0], xmin); ya = max(b[1], ymin)
                xb = min(b[2], xmax); yb = min(b[3], ymax)
                inter = max(0, xb - xa) * max(0, yb - ya)
                area_a = (b[2] - b[0]) * (b[3] - b[1])
                area_b = (xmax - xmin) * (ymax - ymin)
                union = area_a + area_b - inter
                iou = inter / union if union > 0 else 0
                if iou > best_iou:
                    best_iou, best_idx = iou, i
            return masks[best_idx], boxes[best_idx]

        # Default: largest area
        if cfg.auto_select_largest:
            areas = masks.reshape(len(masks), -1).sum(axis=1)
            idx = int(np.argmax(areas))
            return masks[idx], boxes[idx]

    # ------------------------------------------------------------------
    # SUBSEQUENT FRAMES: re-identify by IoU with the previous mask
    # ------------------------------------------------------------------
    ious = np.array([mask_iou(m, prev_mask) for m in masks])
    best = int(np.argmax(ious))
    if ious[best] >= cfg.iou_continuity_thresh:
        return masks[best], boxes[best]

    # Fallback: closest centroid to the previous patient's centroid.
    pcx, pcy = mask_centroid(prev_mask)
    if not math.isnan(pcx):
        dists = []
        for m in masks:
            cx, cy = mask_centroid(m)
            if math.isnan(cx):
                dists.append(np.inf)
            else:
                dists.append((cx - pcx) ** 2 + (cy - pcy) ** 2)
        idx = int(np.argmin(dists))
        # Sanity check: also require non-trivial overlap with previous bbox area
        if not np.isinf(dists[idx]):
            return masks[idx], boxes[idx]

    return None, None

In [ ]:
def get_main_contour(mask: np.ndarray) -> Optional[np.ndarray]:
    """Return the largest external contour (N, 2) in (x, y), or None if mask is empty."""
    cnts, _ = cv2.findContours(
        mask.astype(np.uint8) * 255,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )
    if not cnts:
        return None
    main = max(cnts, key=cv2.contourArea)
    pts  = main.reshape(-1, 2).astype(np.float32)   # (N, 2)
    return pts

def resample_contour(pts: np.ndarray, n: int, anchor_xy: Tuple[float, float]) -> np.ndarray:
    """
    Resample a contour to exactly `n` points evenly spaced by perimeter arc-length.

    The first output point is the contour vertex closest to `anchor_xy`. This stabilizes
    the index ordering across frames (so 'point 0' is consistently the same body region).
    """
    # 1) Reorder the contour so it starts at the vertex closest to the anchor.
    diffs   = pts - np.array(anchor_xy, dtype=np.float32)
    start   = int(np.argmin((diffs ** 2).sum(axis=1)))
    pts     = np.roll(pts, -start, axis=0)

    # 2) Compute cumulative arc length around the contour.
    seg     = np.linalg.norm(np.diff(pts, axis=0, append=pts[:1]), axis=1)
    cumlen  = np.concatenate(([0.0], np.cumsum(seg[:-1])))
    total   = cumlen[-1] + seg[-1]
    if total <= 1e-6:
        return np.repeat(pts[:1], n, axis=0)

    # 3) Sample n target lengths evenly along the perimeter.
    targets = np.linspace(0, total, n, endpoint=False)
    out     = np.zeros((n, 2), dtype=np.float32)
    for k, t in enumerate(targets):
        idx = int(np.searchsorted(cumlen, t, side='right') - 1)
        idx = max(0, min(idx, len(pts) - 2))
        seg_len = seg[idx] if seg[idx] > 1e-6 else 1.0
        alpha   = (t - cumlen[idx]) / seg_len
        out[k]  = pts[idx] * (1 - alpha) + pts[(idx + 1) % len(pts)] * alpha
    return out

def sample_contour_points(mask: np.ndarray, n: int) -> np.ndarray:
    """Top-level: return n evenly-spaced contour points (n, 2). NaN if mask empty."""
    pts = get_main_contour(mask)
    if pts is None or len(pts) < 3:
        return np.full((n, 2), np.nan, dtype=np.float32)

    cx, cy = mask_centroid(mask)
    # Anchor = highest point along the contour above the centroid (≈ head)
    above = pts[pts[:, 1] < cy]
    if len(above) > 0:
        anchor_idx = int(np.argmin(above[:, 1]))
        anchor = (float(above[anchor_idx, 0]), float(above[anchor_idx, 1]))
    else:
        anchor = (cx, cy)
    return resample_contour(pts, n, anchor)

def sample_interior_grid(
    mask: np.ndarray,
    box:  np.ndarray,           # xyxy
    rows: int,
    cols: int,
) -> np.ndarray:
    """
    Sample a `rows × cols` grid inside the bbox of the mask.

    Returns
    -------
    out : (rows*cols, 2) float32
        Grid in row-major order (top→bottom, left→right). Points outside the mask = NaN.
    """
    H, W = mask.shape
    x1, y1, x2, y2 = [float(v) for v in box]

    out = np.full((rows * cols, 2), np.nan, dtype=np.float32)
    if x2 <= x1 or y2 <= y1:
        return out

    for r in range(rows):
        v = r / max(rows - 1, 1)
        py = y1 + v * (y2 - y1)
        for c in range(cols):
            u = c / max(cols - 1, 1)
            px = x1 + u * (x2 - x1)
            ix = int(round(px)); iy = int(round(py))
            if 0 <= ix < W and 0 <= iy < H and mask[iy, ix]:
                out[r * cols + c] = (px, py)
    return out

In [ ]:
def geometric_descriptors(mask: np.ndarray, box: np.ndarray) -> dict:
    """
    Return a dict of geometric descriptors for the body silhouette:
      - centroid_x, centroid_y
      - area_px                (number of mask pixels)
      - perimeter_px           (length of the main contour)
      - bbox_x, bbox_y, bbox_w, bbox_h
      - aspect_ratio           (h / w)
      - solidity               (area / convex_hull_area)
      - extent                 (area / bbox_area)
      - orientation_deg        (angle of principal axis, in [-90, +90])
      - major_axis_px          (length of major axis from PCA)
      - minor_axis_px          (length of minor axis from PCA)
    All values are NaN when the mask is empty."""
    out = {
        'centroid_x': np.nan, 'centroid_y': np.nan,
        'area_px': np.nan, 'perimeter_px': np.nan,
        'bbox_x': np.nan, 'bbox_y': np.nan, 'bbox_w': np.nan, 'bbox_h': np.nan,
        'aspect_ratio': np.nan, 'solidity': np.nan, 'extent': np.nan,
        'orientation_deg': np.nan, 'major_axis_px': np.nan, 'minor_axis_px': np.nan,
    }
    if mask is None or mask.sum() == 0:
        return out

    cx, cy = mask_centroid(mask)
    out['centroid_x'] = cx; out['centroid_y'] = cy
    out['area_px'] = float(mask.sum())

    cnts, _ = cv2.findContours(
        mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    if not cnts:
        return out
    main = max(cnts, key=cv2.contourArea)
    out['perimeter_px'] = float(cv2.arcLength(main, True))

    x1, y1, x2, y2 = [float(v) for v in box]
    out['bbox_x'], out['bbox_y'] = x1, y1
    out['bbox_w'], out['bbox_h'] = x2 - x1, y2 - y1
    if (x2 - x1) > 0:
        out['aspect_ratio'] = (y2 - y1) / (x2 - x1)
        out['extent'] = out['area_px'] / max((x2 - x1) * (y2 - y1), 1)

    hull = cv2.convexHull(main)
    hull_area = cv2.contourArea(hull)
    if hull_area > 0:
        out['solidity'] = out['area_px'] / hull_area

    # Principal axes via PCA on the mask pixels (subsample for speed if very large)
    ys, xs = np.where(mask)
    if len(xs) >= 5:
        if len(xs) > 5000:
            idx = np.random.default_rng(0).choice(len(xs), size=5000, replace=False)
            xs, ys = xs[idx], ys[idx]
        pts = np.stack([xs, ys], axis=1).astype(np.float32)
        pts -= pts.mean(axis=0, keepdims=True)
        cov = np.cov(pts.T)
        evals, evecs = np.linalg.eigh(cov)
        order = np.argsort(evals)[::-1]
        evals, evecs = evals[order], evecs[:, order]
        major_vec = evecs[:, 0]
        out['orientation_deg'] = float(np.degrees(np.arctan2(major_vec[1], major_vec[0])))
        # Convert eigenvalues (variance) to lengths (≈ 4σ covers ~95 % of points)
        out['major_axis_px'] = float(4.0 * np.sqrt(max(evals[0], 0)))
        out['minor_axis_px'] = float(4.0 * np.sqrt(max(evals[1], 0)))
    return out

In [ ]:
import tempfile

def _set_image_robust(predictor, frame_bgr: np.ndarray):
    """Call SAM 3's set_image() whether it expects a path or a numpy array."""
    try:
        predictor.set_image(frame_bgr)            # ndarray path (preferred, fast)
        return None
    except Exception:
        # Fallback: save to a temp file and pass the path.
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
        tmp.close()
        cv2.imwrite(tmp.name, frame_bgr)
        predictor.set_image(tmp.name)
        return tmp.name                           # caller is responsible for cleanup

def sam3_segment_persons(
    frame_bgr: np.ndarray,
    predictor: SAM3SemanticPredictor,
    text:      str = 'person',
) -> Tuple[np.ndarray, np.ndarray]:
    """Run SAM 3 semantic segmentation on a single BGR frame.

    Returns
    -------
    masks : (K, H, W) bool
        One boolean mask per detected instance. Empty array if nothing detected.
    boxes : (K, 4) float32
        xyxy bounding boxes in pixel coordinates of the original frame.
    """
    H, W = frame_bgr.shape[:2]

    tmp_path = _set_image_robust(predictor, frame_bgr)
    try:
        results = predictor(text=[text])
    finally:
        if tmp_path is not None:
            try: os.unlink(tmp_path)
            except Exception: pass

    # Ultralytics may return a list of Results or a single Results object.
    if isinstance(results, list):
        if len(results) == 0:
            return np.zeros((0, H, W), dtype=bool), np.zeros((0, 4), dtype=np.float32)
        r = results[0]
    else:
        r = results

    if r is None or r.masks is None or r.masks.data is None or len(r.masks.data) == 0:
        return np.zeros((0, H, W), dtype=bool), np.zeros((0, 4), dtype=np.float32)

    # masks.data is a torch tensor of shape (K, h, w) at the model's processed resolution.
    masks_t = r.masks.data.detach().cpu().numpy().astype(bool)
    if masks_t.shape[1:] != (H, W):
        # Resize each mask to the original frame size with nearest-neighbour.
        masks_resized = np.zeros((masks_t.shape[0], H, W), dtype=bool)
        for i in range(masks_t.shape[0]):
            mr = cv2.resize(
                masks_t[i].astype(np.uint8), (W, H),
                interpolation=cv2.INTER_NEAREST,
            )
            masks_resized[i] = mr.astype(bool)
        masks = masks_resized
    else:
        masks = masks_t

    if r.boxes is not None and r.boxes.xyxy is not None and len(r.boxes.xyxy) > 0:
        boxes = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    else:
        # Recompute boxes from masks if SAM 3 didn't return them.
        boxes = np.zeros((len(masks), 4), dtype=np.float32)
        for i, m in enumerate(masks):
            ys, xs = np.where(m)
            if len(xs) > 0:
                boxes[i] = [xs.min(), ys.min(), xs.max(), ys.max()]
    return masks, boxes

In [ ]:
def draw_annotation(
    frame_bgr:  np.ndarray,
    mask:       Optional[np.ndarray],
    box:        Optional[np.ndarray],
    contour_pts: Optional[np.ndarray],   # (N, 2)
    grid_pts:    Optional[np.ndarray],   # (M, 2), NaN for outside-mask
    descr:       Optional[dict],
) -> np.ndarray:
    """Return a BGR copy of `frame_bgr` overlaid with patient annotations."""
    out = frame_bgr.copy()
    if mask is not None and mask.any():
        # Translucent magenta tint of the mask
        tint = np.zeros_like(out)
        tint[mask] = (180, 50, 220)        # BGR: pinkish magenta
        out = cv2.addWeighted(out, 1.0, tint, 0.35, 0)

        # Mask contour (white, thin)
        cnts, _ = cv2.findContours(
            mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        if cnts:
            cv2.drawContours(out, cnts, -1, (255, 255, 255), 1, cv2.LINE_AA)

    if box is not None:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(out, 'patient', (x1, max(0, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

    if grid_pts is not None:
        for x, y in grid_pts:
            if not np.isnan(x):
                cv2.circle(out, (int(x), int(y)), 2, (255, 220, 0), -1, cv2.LINE_AA)

    if contour_pts is not None:
        for x, y in contour_pts:
            if not np.isnan(x):
                cv2.circle(out, (int(x), int(y)), 3, (0, 140, 255), -1, cv2.LINE_AA)

    if descr is not None and not np.isnan(descr.get('centroid_x', np.nan)):
        cx, cy = int(descr['centroid_x']), int(descr['centroid_y'])
        cv2.drawMarker(out, (cx, cy), (0, 0, 255), cv2.MARKER_CROSS, 16, 2, cv2.LINE_AA)
    return out

def render_mask_only(
    frame_shape: Tuple[int, int],   # (H, W)
    mask:        Optional[np.ndarray],
    contour_pts: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Black background, white-filled patient silhouette + magenta outline.

    A pure-mask video is convenient for downstream image processing tasks (e.g. computing
    optical flow only inside the patient, or training a model on silhouettes only)."""
    H, W = frame_shape
    out = np.zeros((H, W, 3), dtype=np.uint8)
    if mask is not None and mask.any():
        out[mask] = (240, 240, 240)        # near-white fill
        cnts, _ = cv2.findContours(
            mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        if cnts:
            cv2.drawContours(out, cnts, -1, (180, 50, 220), 2, cv2.LINE_AA)
        if contour_pts is not None:
            for x, y in contour_pts:
                if not np.isnan(x):
                    cv2.circle(out, (int(x), int(y)), 3, (0, 140, 255), -1, cv2.LINE_AA)
    return out

def render_sidebyside(
    raw_bgr:    np.ndarray,
    mask_bgr:   np.ndarray,
    overlay_bgr: np.ndarray,
    labels:     Tuple[str, str, str] = ('original', 'mask', 'overlay'),
) -> np.ndarray:
    """Concatenate the three frames horizontally with text labels on top."""
    H, W = raw_bgr.shape[:2]
    panels = [raw_bgr, mask_bgr, overlay_bgr]
    out = np.concatenate(panels, axis=1)
    for i, label in enumerate(labels):
        cv2.putText(out, label, (i * W + 10, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.line(out, (i * W, 0), (i * W, H), (60, 60, 60), 1)
    return out

In [ ]:
GEOM_KEYS = [
    'centroid_x', 'centroid_y', 'area_px', 'perimeter_px',
    'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h',
    'aspect_ratio', 'solidity', 'extent',
    'orientation_deg', 'major_axis_px', 'minor_axis_px',
]

META_KEYS = ['subject_id', 'subject_type', 'video_relpath']

def build_columns(cfg: Settings) -> List[str]:
    cols = list(META_KEYS) + ['time_s', 'frame_idx', 'patient_detected']
    cols += GEOM_KEYS
    for i in range(cfg.n_contour_points):
        cols += [f'contour_{i:03d}_x', f'contour_{i:03d}_y']
    for r in range(cfg.grid_rows):
        for c in range(cfg.grid_cols):
            cols += [f'grid_r{r:02d}_c{c:02d}_x', f'grid_r{r:02d}_c{c:02d}_y']
    return cols

def empty_row(
    cfg:           Settings,
    time_s:        float,
    frame_idx:     int,
    subject_id:    str = '',
    subject_type:  str = '',
    video_relpath: str = '',
) -> dict:
    row = {
        'subject_id':    subject_id,
        'subject_type':  subject_type,
        'video_relpath': video_relpath,
        'time_s':        time_s,
        'frame_idx':     frame_idx,
        'patient_detected': 0,
    }
    for k in GEOM_KEYS:
        row[k] = np.nan
    for i in range(cfg.n_contour_points):
        row[f'contour_{i:03d}_x'] = np.nan
        row[f'contour_{i:03d}_y'] = np.nan
    for r in range(cfg.grid_rows):
        for c in range(cfg.grid_cols):
            row[f'grid_r{r:02d}_c{c:02d}_x'] = np.nan
            row[f'grid_r{r:02d}_c{c:02d}_y'] = np.nan
    return row

In [ ]:
def process_video_sam3(
    entry:      VideoEntry,
    predictor:  SAM3SemanticPredictor,
    cfg:        Settings,
    output_dir: Path,
    write_overlay:     bool = True,
    write_mask:        bool = True,
    write_sidebyside:  bool = True,
) -> pd.DataFrame:
    """Process one video end-to-end with SAM 3 + dense surface sampling.

    Outputs go into `output_dir/<subject_id>/`. Filenames mirror the input:
      <output_dir>/<subject>/<stem>_sam3_overlay.mp4      annotated overlay
      <output_dir>/<subject>/<stem>_sam3_mask.mp4         silhouette only
      <output_dir>/<subject>/<stem>_sam3_sidebyside.mp4   raw | mask | overlay
      <output_dir>/<subject>/<stem>_sam3_timeseries.xlsx  per-frame features

    The Excel includes columns subject_id / subject_type / video_relpath so that all
    per-video files concatenate cleanly into a single TabICLv2-ready table.
    """
    file_name = entry.path
    cap = cv2.VideoCapture(str(file_name))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {file_name}')

    fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    stem        = file_name.stem
    subject_dir = output_dir / entry.subject_id
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_overlay     = subject_dir / f'{stem}_sam3_overlay.mp4'
    out_mask        = subject_dir / f'{stem}_sam3_mask.mp4'
    out_sidebyside  = subject_dir / f'{stem}_sam3_sidebyside.mp4'
    out_excel_path  = subject_dir / f'{stem}_sam3_timeseries.xlsx'

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer_overlay = (
        cv2.VideoWriter(str(out_overlay), fourcc, fps, (width, height))
        if write_overlay else None
    )
    writer_mask = (
        cv2.VideoWriter(str(out_mask), fourcc, fps, (width, height))
        if write_mask else None
    )
    writer_sbs = (
        cv2.VideoWriter(str(out_sidebyside), fourcc, fps, (width * 3, height))
        if write_sidebyside else None
    )

    # Apply manual override on this video, if present in the CSV.
    cfg_local = Settings(**{**cfg.__dict__, 'manual_bbox': entry.manual_bbox})

    cols = build_columns(cfg_local)
    rows: List[dict] = []

    prev_mask: Optional[np.ndarray] = None
    frame_idx = -1
    t0 = time.time()

    target_mode = 'MANUAL' if entry.manual_bbox is not None else 'auto'
    print(f'[{entry.subject_id}/{stem}] {nframes} frames, {fps:.1f} fps, '
          f'{width}x{height}  target={target_mode} -> start')

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1
        time_s = round(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0, 3)

        if cfg_local.frame_stride > 1 and (frame_idx % cfg_local.frame_stride) != 0:
            continue

        # ---- 1) SAM 3 segmentation ----
        masks, boxes = sam3_segment_persons(frame, predictor, cfg_local.text_prompt)

        # ---- 2) patient selection (ignores all other persons) ----
        mask_sel, box_sel = select_patient_mask(masks, boxes, prev_mask, cfg_local)

        meta = dict(
            subject_id    = entry.subject_id,
            subject_type  = entry.subject_type,
            video_relpath = entry.relative_path,
        )

        if mask_sel is None or not mask_sel.any():
            row = empty_row(cfg_local, time_s, frame_idx, **meta)
            rows.append(row)
            empty_mask_frame = np.zeros_like(frame)
            if writer_overlay is not None: writer_overlay.write(frame)
            if writer_mask    is not None: writer_mask.write(empty_mask_frame)
            if writer_sbs     is not None:
                writer_sbs.write(render_sidebyside(frame, empty_mask_frame, frame))
            continue

        prev_mask = mask_sel

        # ---- 3) sampling: contour + interior grid ----
        contour_pts = sample_contour_points(mask_sel, cfg_local.n_contour_points)
        grid_pts    = sample_interior_grid(
            mask_sel, box_sel, cfg_local.grid_rows, cfg_local.grid_cols
        )

        # ---- 4) geometric descriptors ----
        descr = geometric_descriptors(mask_sel, box_sel)

        # ---- 5) build row ----
        row = empty_row(cfg_local, time_s, frame_idx, **meta)
        row['patient_detected'] = 1
        for k in GEOM_KEYS:
            row[k] = descr[k]
        for i, (x, y) in enumerate(contour_pts):
            row[f'contour_{i:03d}_x'] = float(x) if not np.isnan(x) else np.nan
            row[f'contour_{i:03d}_y'] = float(y) if not np.isnan(y) else np.nan
        idx = 0
        for r in range(cfg_local.grid_rows):
            for c in range(cfg_local.grid_cols):
                gx, gy = grid_pts[idx]
                row[f'grid_r{r:02d}_c{c:02d}_x'] = float(gx) if not np.isnan(gx) else np.nan
                row[f'grid_r{r:02d}_c{c:02d}_y'] = float(gy) if not np.isnan(gy) else np.nan
                idx += 1
        rows.append(row)

        # ---- 6) annotated frames (three views) ----
        overlay_frame = draw_annotation(frame, mask_sel, box_sel, contour_pts, grid_pts, descr)
        mask_frame    = render_mask_only((height, width), mask_sel, contour_pts)
        if writer_overlay is not None: writer_overlay.write(overlay_frame)
        if writer_mask    is not None: writer_mask.write(mask_frame)
        if writer_sbs     is not None:
            writer_sbs.write(render_sidebyside(frame, mask_frame, overlay_frame))

        if frame_idx % 50 == 0 and frame_idx > 0:
            elapsed = time.time() - t0
            rate    = (frame_idx + 1) / elapsed
            eta     = (nframes - frame_idx - 1) / max(rate, 1e-6)
            print(f'  [{frame_idx:5d}/{nframes}]  {rate:.2f} fps  ETA {eta/60:.1f} min')

    cap.release()
    if writer_overlay is not None: writer_overlay.release()
    if writer_mask    is not None: writer_mask.release()
    if writer_sbs     is not None: writer_sbs.release()

    df = pd.DataFrame(rows, columns=cols)
    df.to_excel(out_excel_path, index=False)

    n_det = int(df.patient_detected.sum())
    print(f'[{entry.subject_id}/{stem}] done in {(time.time() - t0)/60:.1f} min  '
          f'(detected on {n_det}/{len(df)} frames)')
    return df

## 10. Anti-idle (avant les longues extractions)

In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function ClickConnect(){
  const b = document.querySelector('#top-toolbar > colab-connect-button');
  if (b && b.shadowRoot) { const c = b.shadowRoot.querySelector('#connect'); if (c) c.click(); }
}
setInterval(ClickConnect, 60000);
'''))
print('Anti-idle actif.')

## 11. Run the extraction on ONE dataset

Pick the dataset (`1`, `2` or `3`). Each dataset writes to its own output folder to avoid any collision (P1 of dataset_2 is not P1 of dataset_3).

Files (annotated videos + `_sam3_timeseries.xlsx`) are written **directly to Drive** -> they persist if Colab disconnects. **Resumable**: videos whose `.xlsx` already exists are skipped.

In [ ]:
# ===== CHOISIR LE DATASET ICI =====
DATASET_N = 1   # 1, 2 ou 3
# ==================================

dataset_dir = INFER_ROOT / f'dataset_{DATASET_N}'
output_dir  = OUTPUT_ROOT / f'dataset_{DATASET_N}'
output_dir.mkdir(parents=True, exist_ok=True)

assert dataset_dir.is_dir(), f'Introuvable: {dataset_dir}'

# Decouverte
video_entries = discover_dataset(dataset_dir, manual_targets={})
print_dataset_summary(video_entries)

In [ ]:
# Extraction (resumable: skips videos already processed)
results_summary = []
for k, entry in enumerate(video_entries, 1):
    stem = entry.path.stem
    out_xlsx = output_dir / entry.subject_id / f'{stem}_sam3_timeseries.xlsx'
    if out_xlsx.exists():
        print(f'[{k}/{len(video_entries)}] SKIP {entry.relative_path} (already done)')
        results_summary.append((entry.relative_path, 'skipped'))
        continue
    print(f'\n[{k}/{len(video_entries)}] === {entry.relative_path} ===')
    try:
        df = process_video_sam3(
            entry, predictor, S, output_dir,
            write_overlay=True, write_mask=True, write_sidebyside=True,
        )
        results_summary.append((entry.relative_path, f'ok ({len(df)} frames)'))
    except Exception as e:
        print(f'   [ERREUR] {entry.relative_path}: {e}')
        results_summary.append((entry.relative_path, f'ERROR: {e}'))

print('\n===== RESUME =====')
for relpath, status in results_summary:
    print(f'  {relpath:<30s} {status}')

## 12. Output check

In [ ]:
# Compter les timeseries produits pour ce dataset
xlsx_files = sorted(output_dir.rglob('*_sam3_timeseries.xlsx'))
print(f'dataset_{DATASET_N}: {len(xlsx_files)} timeseries files')
for f in xlsx_files:
    df = pd.read_excel(f)
    det = int(df.patient_detected.sum())
    print(f'  {f.parent.name}/{f.name}: {len(df)} frames, patient detecte sur {det}')

## Next steps

1. Re-run cell 11 with `DATASET_N = 2`, then `DATASET_N = 3`.
2. Once the 3 datasets are processed, download `outputs_inference/` from Drive to your PC (or keep it on Drive).
3. Run the external validation (per-patient aggregation + Tier 2 inference + multi-consensus evaluation): see `phase_3_external_validation_v1_manual.ipynb`.

Note: the 3D section (SAM 3D Body) is NOT included here — it is separate and not needed for phenomenology inference.